# Betting simulation notebook

In [1]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Literal

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
Strategy = Literal[
    "all_in",
    "fixed_amount",
    "proportion",
    "martingale",
    "reversed_martingale",
    "fibonacci",
    "sequence_1326",
]

Status = Literal["win", "loss"]

STRATEGY_LABELS: dict[str, str] = {
    "all_in": "All-in",
    "fixed_amount": "Fixed amount",
    "proportion": "Proportion",
    "martingale": "Martingale",
    "reversed_martingale": "Reverse martingale",
    "fibonacci": "Fibonacci",
    "sequence_1326": "1-3-2-6",
}

STRATEGIES: tuple[Strategy, ...] = (
    "all_in",
    "fixed_amount",
    "proportion",
    "martingale",
    "reversed_martingale",
    "fibonacci",
    "sequence_1326",
)

SEED = 8888

OUTLIER_TRIM_PCT = 0.02
OUTLIER_TRIM_VALUE_COLUMN = "profit"
OUTLIER_TRIM_GROUP_COLUMNS = ("strategy",)

PLOTLY_OUTPUT_DIR = Path("data/plotly")
PLOTLY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## The simulator

In [3]:
@dataclass(frozen=True)
class BetRecord:
    """One realised betting round."""

    round_no: int
    bankroll: float
    bet_amount: float
    status: str | None
    profit: float


@dataclass(frozen=True)
class BetSummary:
    """Summary statistics for one simulation run."""

    strategy: str
    rounds_played: int
    wins: int
    losses: int
    final_bankroll: float
    profit: float
    win_rate: float
    ruined: bool


class Bettor:
    """Simulate one bettor following a fixed betting strategy.

    Parameters
    ----------
    strategy:
        Name of the betting strategy to use.
    initial_bankroll:
        Starting bankroll.
    win_probability:
        Probability that a single bet wins.
    commission:
        Fraction removed from the gross payout when the bet wins.
        For example, with `payout_multiple=2` and `commission=0.03`,
        a winning 100 bet returns 194, not 200.
    minimum_bet:
        Smallest allowed bet. The bettor stops when the bankroll is below this.
    maximum_bet:
        Table limit. This is what makes systems such as martingale break.
    payout_multiple:
        Gross payout multiple on a winning bet, including the original stake.
    proportion:
        Fraction of bankroll used by the proportional strategy.
    rng:
        Optional NumPy random generator, useful for reproducible simulations.
    """

    def __init__(
        self,
        strategy: Strategy = "proportion",
        initial_bankroll: float = 1_000.0,
        win_probability: float = 0.5,
        commission: float = 0.0,
        minimum_bet: float = 1.0,
        maximum_bet: float = 5_000.0,
        payout_multiple: float = 2.0,
        fixed_amount: float = 1.0,
        proportion: float = 0.01,
        martingale_multiplier: float = 2.0,
        reverse_martingale_multiplier: float = 2.0,
        reverse_martingale_continue_probability: float = 0.5,
        rng: np.random.Generator | None = None,
    ) -> None:
        self.strategy = strategy
        self.initial_bankroll = float(initial_bankroll)
        self.bankroll = float(initial_bankroll)
        self.win_probability = float(win_probability)
        self.commission = float(commission)
        self.minimum_bet = float(minimum_bet)
        self.maximum_bet = float(maximum_bet)
        self.payout_multiple = float(payout_multiple)
        self.fixed_amount = float(fixed_amount)
        self.proportion = float(proportion)
        self.martingale_multiplier = float(martingale_multiplier)
        self.reverse_martingale_multiplier = float(reverse_martingale_multiplier)
        self.reverse_martingale_continue_probability = float(reverse_martingale_continue_probability)
        self.rng = rng or np.random.default_rng()

        self.wins = 0
        self.losses = 0
        self.last_status: Status | None = None
        self.last_bet_amount = 0.0
        self.records: list[BetRecord] = [
            BetRecord(
                round_no=0,
                bankroll=self.initial_bankroll,
                bet_amount=0.0,
                status=None,
                profit=0.0,
            )
        ]

    @property
    def rounds_played(self) -> int:
        """Number of bets already placed."""
        return self.wins + self.losses

    def can_bet(self) -> bool:
        """Return whether the bettor has enough bankroll to place a bet."""
        return self.bankroll >= self.minimum_bet

    def cap_bet_amount(self, requested_amount: float) -> float:
        """Apply the minimum bet, table limit, and bankroll limit."""
        requested_amount = max(float(requested_amount), self.minimum_bet)
        return min(requested_amount, self.maximum_bet, self.bankroll)

    def win_gross_return(self, bet_amount: float) -> float:
        """Return the gross cash credited when a bet wins."""
        return self.payout_multiple * bet_amount * (1.0 - self.commission)

    def draw_result(self) -> Status:
        """Draw the result for one bet."""
        return "win" if self.rng.random() < self.win_probability else "loss"

    def next_bet_amount(self) -> float:
        """Return the requested amount before table and bankroll limits are applied."""
        if self.strategy == "all_in":
            return self.bankroll

        if self.strategy == "fixed_amount":
            return self.fixed_amount

        if self.strategy == "proportion":
            return self.proportion * self.bankroll

        if self.strategy == "martingale":
            if self.last_status in (None, "win"):
                return self.minimum_bet
            return self.martingale_multiplier * self.last_bet_amount

        if self.strategy == "reversed_martingale":
            if self.last_status in (None, "loss"):
                return self.minimum_bet
            should_press = self.rng.random() < self.reverse_martingale_continue_probability
            if should_press:
                return self.reverse_martingale_multiplier * self.last_bet_amount
            return self.minimum_bet

        if self.strategy == "fibonacci":
            if len(self.records) < 3:
                return self.minimum_bet
            last_two = self.records[-2:]
            if all(record.status == "loss" for record in last_two):
                return last_two[0].bet_amount + last_two[1].bet_amount
            return self.minimum_bet

        if self.strategy == "sequence_1326":
            sequence = (1, 3, 2, 6)
            if self.last_status in (None, "loss"):
                return self.minimum_bet
            previous_multiplier = round(self.last_bet_amount / self.minimum_bet)
            try:
                previous_index = sequence.index(previous_multiplier)
            except ValueError:
                return self.minimum_bet
            next_multiplier = sequence[(previous_index + 1) % len(sequence)]
            return next_multiplier * self.minimum_bet

        raise ValueError(f"Unknown strategy: {self.strategy}")

    def play_round(self) -> bool:
        """Place and settle one bet.

        Returns
        -------
        bool
            `True` when a bet was placed, `False` when the bettor has stopped.
        """
        if not self.can_bet():
            return False

        bet_amount = self.cap_bet_amount(self.next_bet_amount())
        self.bankroll -= bet_amount

        status = self.draw_result()
        if status == "win":
            self.bankroll += self.win_gross_return(bet_amount)
            self.wins += 1
        else:
            self.losses += 1

        self.last_status = status
        self.last_bet_amount = bet_amount
        self.records.append(
            BetRecord(
                round_no=self.rounds_played,
                bankroll=self.bankroll,
                bet_amount=bet_amount,
                status=status,
                profit=self.bankroll - self.initial_bankroll,
            )
        )
        return True

    def run(self, rounds: int = 10_000) -> "Bettor":
        """Run the simulation until the round limit or bankroll limit is reached."""
        for _ in range(rounds):
            if not self.play_round():
                break
        return self

    def to_frame(self) -> pd.DataFrame:
        """Return the betting records as a DataFrame."""
        return pd.DataFrame(asdict(record) for record in self.records)

    def summary(self) -> BetSummary:
        """Return summary statistics for the bettor."""
        rounds_played = self.rounds_played
        win_rate = self.wins / rounds_played if rounds_played else 0.0
        return BetSummary(
            strategy=self.strategy,
            rounds_played=rounds_played,
            wins=self.wins,
            losses=self.losses,
            final_bankroll=self.bankroll,
            profit=self.bankroll - self.initial_bankroll,
            win_rate=win_rate,
            ruined=not self.can_bet(),
        )

## Helpers

In [4]:
def run_single_strategy(
    strategy: Strategy,
    rounds: int = 1_000,
    seed: int = 1,
    **bettor_kwargs: float,
) -> tuple[pd.DataFrame, BetSummary]:
    """Run one bettor and return the record table plus summary."""
    bettor = Bettor(
        strategy=strategy,
        rng=np.random.default_rng(seed),
        **bettor_kwargs,
    ).run(rounds=rounds)
    return bettor.to_frame(), bettor.summary()


def simulate_strategy(
    strategy: Strategy,
    n_simulations: int = 500,
    rounds: int = 1_000,
    seed: int = 42,
    **bettor_kwargs: float,
) -> pd.DataFrame:
    """Run many independent bettors using the same strategy."""
    rng = np.random.default_rng(seed)
    rows: list[dict[str, float | int | str | bool]] = []

    for simulation_no in range(1, n_simulations + 1):
        bettor_seed = int(rng.integers(0, np.iinfo(np.int32).max))
        bettor = Bettor(
            strategy=strategy,
            rng=np.random.default_rng(bettor_seed),
            **bettor_kwargs,
        ).run(rounds=rounds)
        summary = asdict(bettor.summary())
        summary["simulation_no"] = simulation_no
        summary["strategy_label"] = STRATEGY_LABELS[strategy]
        rows.append(summary)

    return pd.DataFrame(rows)


def simulate_many_strategies(
    strategies: tuple[Strategy, ...] = STRATEGIES,
    n_simulations: int = 500,
    rounds: int = 1_000,
    seed: int = 42,
    **bettor_kwargs: float,
) -> pd.DataFrame:
    """Run a strategy comparison table."""
    frames = []
    for offset, strategy in enumerate(strategies):
        frames.append(
            simulate_strategy(
                strategy=strategy,
                n_simulations=n_simulations,
                rounds=rounds,
                seed=seed + offset,
                **bettor_kwargs,
            )
        )
    return pd.concat(frames, ignore_index=True)


def validate_trim_pct(trim_pct: float) -> float:
    """Validate the trim percentage used to remove both tails of a distribution.

    The value is expressed as a fraction, so `0.02` means removing the bottom 2%
    and top 2% from each group.
    """
    trim_pct = float(trim_pct)
    if not 0 <= trim_pct < 0.5:
        raise ValueError("trim_pct must be in the interval [0, 0.5).")
    return trim_pct


def trim_simulation_outliers(
    simulations: pd.DataFrame,
    trim_pct: float = OUTLIER_TRIM_PCT,
    value_column: str = OUTLIER_TRIM_VALUE_COLUMN,
    group_columns: tuple[str, ...] = OUTLIER_TRIM_GROUP_COLUMNS,
) -> pd.DataFrame:
    """Remove the bottom/top tail observations from simulation results.

    Trimming is done within each strategy by default. This is important because
    strategies have very different payoff scales; global trimming would mostly
    remove observations from the naturally high-variance strategies.

    Parameters
    ----------
    simulations:
        Raw simulation output from `simulate_many_strategies`.
    trim_pct:
        Fraction removed from each tail of `value_column` per group. For example,
        `0.01`, `0.02`, and `0.03` remove the bottom/top 1%, 2%, and 3%.
    value_column:
        Metric used to define outliers. Usually `profit`.
    group_columns:
        Columns that define independent trimming groups. The default trims each
        strategy separately.
    """
    trim_pct = validate_trim_pct(trim_pct)
    if trim_pct == 0:
        return simulations.copy()

    missing_columns = {value_column, *group_columns} - set(simulations.columns)
    if missing_columns:
        raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

    grouped_values = simulations.groupby(list(group_columns), sort=False)[value_column]
    lower_bounds = grouped_values.transform(lambda values: values.quantile(trim_pct))
    upper_bounds = grouped_values.transform(lambda values: values.quantile(1 - trim_pct))

    mask = simulations[value_column].between(lower_bounds, upper_bounds, inclusive="both")
    return simulations.loc[mask].copy().reset_index(drop=True)


def make_trim_report(
    raw: pd.DataFrame,
    trimmed: pd.DataFrame,
    group_columns: tuple[str, ...] = OUTLIER_TRIM_GROUP_COLUMNS,
) -> pd.DataFrame:
    """Return how many simulations were removed by outlier trimming."""
    raw_count = raw.groupby(list(group_columns)).size().rename("raw_count")
    trimmed_count = trimmed.groupby(list(group_columns)).size().rename("trimmed_count")

    report = pd.concat([raw_count, trimmed_count], axis=1).fillna(0).reset_index()
    report["removed_count"] = report["raw_count"] - report["trimmed_count"]
    report["removed_pct"] = report["removed_count"] / report["raw_count"]

    if "strategy" in report.columns:
        report["strategy_label"] = report["strategy"].map(STRATEGY_LABELS)

    display_columns = [
        column for column in ["strategy", "strategy_label", "raw_count", "trimmed_count", "removed_count", "removed_pct"]
        if column in report.columns
    ]
    return report[display_columns]


def summarise_simulations(simulations: pd.DataFrame) -> pd.DataFrame:
    """Aggregate repeated simulations into one row per strategy."""
    summary = (
        simulations
        .groupby(["strategy", "strategy_label"], as_index=False)
        .agg(
            average_profit=("profit", "mean"),
            median_profit=("profit", "median"),
            chance_of_profit=("profit", lambda s: (s > 0).mean()),
            ruin_rate=("ruined", "mean"),
            average_rounds=("rounds_played", "mean"),
            average_win_rate=("win_rate", "mean"),
        )
    )
    return summary.sort_values("average_profit", ascending=False)

## Plotly helpers

The figures are Plotly objects, so they can be displayed in the notebook or exported as standalone HTML files.

In [5]:
def plot_single_runs(single_runs: dict[str, pd.DataFrame]) -> go.Figure:
    """Plot bankroll paths from a few individual strategy runs."""
    fig = go.Figure()
    for strategy, records in single_runs.items():
        fig.add_trace(
            go.Scatter(
                x=records["round_no"],
                y=records["bankroll"],
                mode="lines",
                name=STRATEGY_LABELS[strategy],
                hovertemplate="Round %{x}<br>Bankroll %{y:.2f}<extra></extra>",
            )
        )
    fig.update_layout(
        title="Example bankroll paths from one simulated bettor per strategy",
        xaxis_title="Round",
        yaxis_title="Bankroll",
        hovermode="x unified",
        template="plotly_white",
        height=520,
    )
    return fig


def plot_profit_distribution(simulations: pd.DataFrame) -> go.Figure:
    """Plot final-profit distributions by strategy."""
    fig = px.box(
        simulations,
        x="strategy_label",
        y="profit",
        points=False,
        category_orders={"strategy_label": [STRATEGY_LABELS[s] for s in STRATEGIES]},
        title="Distribution of final profit after repeated simulations, trimmed by strategy",
        labels={"strategy_label": "Strategy", "profit": "Final profit"},
        template="plotly_white",
    )
    fig.update_layout(height=520, xaxis_tickangle=-25)
    return fig


def plot_strategy_summary(summary: pd.DataFrame) -> go.Figure:
    """Plot average and median profit by strategy."""
    ordered = summary.sort_values("median_profit", ascending=False)
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=ordered["strategy_label"],
            y=ordered["average_profit"],
            name="Average profit",
            hovertemplate="%{x}<br>Average profit %{y:.2f}<extra></extra>",
        )
    )
    fig.add_trace(
        go.Bar(
            x=ordered["strategy_label"],
            y=ordered["median_profit"],
            name="Median profit",
            hovertemplate="%{x}<br>Median profit %{y:.2f}<extra></extra>",
        )
    )
    fig.update_layout(
        title="Average and median profit by strategy after outlier trimming",
        xaxis_title="Strategy",
        yaxis_title="Profit",
        barmode="group",
        template="plotly_white",
        height=520,
        xaxis_tickangle=-25,
    )
    return fig


def plot_commission_impact(
    no_commission: pd.DataFrame,
    with_commission: pd.DataFrame,
) -> go.Figure:
    """Compare strategy profits with and without commission.

    Pass trimmed simulation tables if the comparison should exclude extreme tails.
    """
    a = summarise_simulations(no_commission).assign(scenario="No commission")
    b = summarise_simulations(with_commission).assign(scenario="3% commission")
    combined = pd.concat([a, b], ignore_index=True)
    fig = px.bar(
        combined,
        x="strategy_label",
        y="median_profit",
        color="scenario",
        barmode="group",
        category_orders={"strategy_label": [STRATEGY_LABELS[s] for s in STRATEGIES]},
        title="Commission turns a fair-looking game into a losing game, after trimming",
        labels={"strategy_label": "Strategy", "median_profit": "Median profit", "scenario": "Scenario"},
        template="plotly_white",
    )
    fig.update_layout(height=520, xaxis_tickangle=-25)
    return fig

## One run per strategy

A single path is not proof of anything, but it is useful for intuition. It shows how different strategies move the bankroll around.

In [6]:
ROUNDS = 1_000
N_SIMULATIONS = 10_000

single_runs = {}
single_summaries = []
for seed, strategy in enumerate(STRATEGIES, start=100):
    records, summary = run_single_strategy(strategy, rounds=ROUNDS, seed=seed)
    single_runs[strategy] = records
    single_summaries.append(asdict(summary) | {"strategy_label": STRATEGY_LABELS[strategy]})

single_summary_df = pd.DataFrame(single_summaries)
single_summary_df

,strategy,rounds_played,wins,losses,final_bankroll,profit,win_rate,ruined,strategy_label
0,all_in,1,0,1,0.000000,-1000.000000,0.00000,True,All-in
1,fixed_amount,1000,527,473,1054.000000,54.000000,0.52700,False,Fixed amount
2,proportion,1000,482,518,663.640627,-336.359373,0.48200,False,Proportion
3,martingale,638,323,315,0.000000,-1000.000000,0.50627,True,Martingale
4,reversed_martingale,1000,501,499,1014.000000,14.000000,0.50100,False,Reverse martingale
5,fibonacci,1000,528,472,1069.000000,69.000000,0.52800,False,Fibonacci
6,sequence_1326,1000,495,505,1004.000000,4.000000,0.49500,False,1-3-2-6


In [7]:
fig_single_runs = plot_single_runs(single_runs)
fig_single_runs.write_html(PLOTLY_OUTPUT_DIR / "betting_single_run_nav.html", include_plotlyjs="cdn", full_html=True)
fig_single_runs

## Multiple simulations

In [8]:
simulations_no_commission = simulate_many_strategies(
    n_simulations=N_SIMULATIONS,
    rounds=ROUNDS,
    commission=0.0,
    seed=SEED,
)

simulations_no_commission_trimmed = trim_simulation_outliers(
    simulations_no_commission,
    trim_pct=OUTLIER_TRIM_PCT,
)
trim_report_no_commission = make_trim_report(
    raw=simulations_no_commission,
    trimmed=simulations_no_commission_trimmed,
)
summary_no_commission = summarise_simulations(simulations_no_commission_trimmed)
summary_no_commission

,strategy,strategy_label,average_profit,median_profit,chance_of_profit,ruin_rate,average_rounds,average_win_rate
2,fixed_amount,Fixed amount,-0.155542,0.000000,0.482741,0.000000,1000.000000,0.499922
6,sequence_1326,1-3-2-6,-0.390729,-1.000000,0.493021,0.000000,1000.000000,0.499882
1,fibonacci,Fibonacci,-3.174006,40.000000,0.692661,0.023445,988.834760,0.498653
5,reversed_martingale,Reverse martingale,-5.239821,-10.000000,0.431428,0.000000,1000.000000,0.500263
3,martingale,Martingale,-7.506057,483.000000,0.607146,0.276087,851.224473,0.492473
4,proportion,Proportion,-9.924032,-48.772954,0.438229,0.000000,1000.000000,0.500081
0,all_in,All-in,-1000.000000,-1000.000000,0.000000,1.000000,6.812984,0.279122


In [9]:
trim_report_no_commission

,strategy,strategy_label,raw_count,trimmed_count,removed_count,removed_pct
0,all_in,All-in,10000,9951,49,0.0049
1,fibonacci,Fibonacci,10000,9810,190,0.0190
2,fixed_amount,Fixed amount,10000,9618,382,0.0382
3,martingale,Martingale,10000,9823,177,0.0177
4,proportion,Proportion,10000,9600,400,0.0400
5,reversed_martingale,Reverse martingale,10000,9603,397,0.0397
6,sequence_1326,1-3-2-6,10000,9600,400,0.0400


In [10]:
fig_profit_distribution = plot_profit_distribution(simulations_no_commission_trimmed)
fig_profit_distribution.write_html(PLOTLY_OUTPUT_DIR / "betting_profit_distribution.html", include_plotlyjs="cdn", full_html=True)
fig_profit_distribution

In [11]:
fig_strategy_summary = plot_strategy_summary(summary_no_commission)
fig_strategy_summary.write_html(PLOTLY_OUTPUT_DIR / "betting_strategy_summary.html", include_plotlyjs="cdn", full_html=True)
fig_strategy_summary

## Adding commission

Commission is a simple way to model the house edge. Even a small commission pulls the distribution downward. Here, we use 3% commission as example.

In [12]:
simulations_with_commission = simulate_many_strategies(
    n_simulations=N_SIMULATIONS,
    rounds=ROUNDS,
    commission=0.03,
    seed=SEED,
)

simulations_with_commission_trimmed = trim_simulation_outliers(
    simulations_with_commission,
    trim_pct=OUTLIER_TRIM_PCT,
)
trim_report_with_commission = make_trim_report(
    raw=simulations_with_commission,
    trimmed=simulations_with_commission_trimmed,
)
summary_with_commission = summarise_simulations(simulations_with_commission_trimmed)
summary_with_commission

,strategy,strategy_label,average_profit,median_profit,chance_of_profit,ruin_rate,average_rounds,average_win_rate
2,fixed_amount,Fixed amount,-30.150875,-30.000000,0.151175,0.000000,1000.000000,0.499922
5,reversed_martingale,Reverse martingale,-49.800223,-54.480000,0.193021,0.000000,1000.000000,0.500261
1,fibonacci,Fibonacci,-59.523521,-15.850000,0.417347,0.026122,987.556531,0.498591
6,sequence_1326,1-3-2-6,-60.338619,-60.930000,0.197187,0.000000,1000.000000,0.499883
3,martingale,Martingale,-134.458659,312.830000,0.606122,0.293980,841.542449,0.492175
4,proportion,Proportion,-266.146911,-293.279812,0.117868,0.000000,1000.000000,0.500075
0,all_in,All-in,-1000.000000,-1000.000000,0.000000,1.000000,6.033050,0.282588


In [13]:
trim_report_with_commission

,strategy,strategy_label,raw_count,trimmed_count,removed_count,removed_pct
0,all_in,All-in,10000,9985,15,0.0015
1,fibonacci,Fibonacci,10000,9800,200,0.0200
2,fixed_amount,Fixed amount,10000,9618,382,0.0382
3,martingale,Martingale,10000,9800,200,0.0200
4,proportion,Proportion,10000,9604,396,0.0396
5,reversed_martingale,Reverse martingale,10000,9600,400,0.0400
6,sequence_1326,1-3-2-6,10000,9600,400,0.0400


In [14]:
fig_commission_impact = plot_commission_impact(
    no_commission=simulations_no_commission_trimmed,
    with_commission=simulations_with_commission_trimmed,
)
fig_commission_impact.write_html(PLOTLY_OUTPUT_DIR / "betting_commission_impact.html", include_plotlyjs="cdn", full_html=True)
fig_commission_impact